<a href="https://colab.research.google.com/github/hakeee0331/movie-audience-prediction/blob/Common-Data-Preprocessing/%EA%B3%B5%ED%86%B5_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%A0%84%EC%B2%98%EB%A6%AC(%ED%9D%A5%EB%B3%B4%EC%9C%84).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. 전처리 공통 코드
2. TRANSFORMER(시놉시스) 임베딩 추출
3. 모델
* 시놉시스랑 cnn 제외한 그냥 MLP 모델
* 시놉시스 + MLP
* CNN + 시놉시스 + MLP


# #0. Import
시드 고정 = 모델을 여러 번 실행해도 가능한 한 같은 결과가 나오게 만드는 설정


In [ ]:
# ============================================================
# STEP 0. 공통 데이터 전처리 환경 설정
# ============================================================

print("STEP 0 시작")

import os
import re
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1) Google Drive mount
# ------------------------------------------------------------

IS_COLAB = False

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IS_COLAB = True
except ModuleNotFoundError:
    print("Google Colab 환경이 아니므로 Drive mount를 건너뜁니다.")


# ------------------------------------------------------------
# 2) 기본 경로 설정
# ------------------------------------------------------------

if IS_COLAB:
    BASE_DIR = Path("/content/drive/MyDrive/흥보위")
    DATA_DIR = BASE_DIR / "sql_result"
    COMMON_DIR = BASE_DIR / "공통 데이터 전처리"
else:
    data_dir_candidates = [
        Path("data/processed"),
        Path("../../../data/processed"),
    ]
    DATA_DIR = next((path for path in data_dir_candidates if path.exists()), None)
    if DATA_DIR is None:
        raise FileNotFoundError("data/processed 경로를 찾을 수 없습니다. 노트북 실행 위치를 확인하세요.")
    BASE_DIR = DATA_DIR.parents[1]
    COMMON_DIR = DATA_DIR / "common_preprocessing"

# 모델별 결과 저장 폴더에서 재사용 가능하도록 별도 생성
COMMON_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("COMMON_DIR:", COMMON_DIR)


# ------------------------------------------------------------
# 3) Random seed 고정
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("SEED:", SEED)


# ------------------------------------------------------------
# 4) 분석 설정
# ------------------------------------------------------------

START_DATE = pd.Timestamp("2016-01-01")
END_DATE = pd.Timestamp("2026-12-31")

# 흥행 등급 기준
CLASS_BINS = {
    0: "1만 미만",
    1: "1만~10만",
    2: "10만~100만",
    3: "100만 이상"
}

# 누수 feature 사용 여부
# 개봉 예정작 예측이 목적이므로 False 유지
USE_LEAKAGE_FEATURES = False

print("분석 기간:", START_DATE.date(), "~", END_DATE.date())
print("CLASS_BINS:", CLASS_BINS)
print("USE_LEAKAGE_FEATURES:", USE_LEAKAGE_FEATURES)


# ------------------------------------------------------------
# 5) 최종 저장 파일 경로
# ------------------------------------------------------------

PROCESSED_DATA_PATH = COMMON_DIR / "movie_common_processed.csv"
FEATURE_INFO_PATH = COMMON_DIR / "movie_common_feature_info.pkl"
PREPROCESS_LOG_PATH = COMMON_DIR / "movie_common_preprocess_log.json"

print("\n저장 예정 파일:")
print("PROCESSED_DATA_PATH:", PROCESSED_DATA_PATH)
print("FEATURE_INFO_PATH:", FEATURE_INFO_PATH)
print("PREPROCESS_LOG_PATH:", PREPROCESS_LOG_PATH)

print("\nSTEP 0 완료")

# #1. 데이터 불러오기 / 컬럼명 통일
* ? 배우 점수랑 감독 점수는 무엇인가? > 해결


In [ ]:
# ============================================================
# STEP 1 수정판. 원본 CSV 로드 및 컬럼명 통일
# ============================================================

print("STEP 1 수정판 시작")

# ------------------------------------------------------------
# 1) 원본 CSV 파일 지정
# ------------------------------------------------------------

raw_csv_path = DATA_DIR / "kobis_kmdb_with_synopsis_score.csv"

if not raw_csv_path.exists():
    raise FileNotFoundError(f"원본 CSV 파일을 찾을 수 없습니다: {raw_csv_path}")

print("\n선택된 원본 CSV:")
print(raw_csv_path)


# ------------------------------------------------------------
# 2) CSV 로드
# ------------------------------------------------------------

df_raw = pd.read_csv(raw_csv_path)

print("\n원본 df_raw shape:", df_raw.shape)
print("\n원본 컬럼:")
print(df_raw.columns.tolist())

display(df_raw.head())


# ------------------------------------------------------------
# 3) 컬럼명 통일
# ------------------------------------------------------------

rename_map = {
    "movie_name_clean": "영화명",
    "release_date": "개봉일",
    "cumulative_audience": "누적관객수",
    "show_count": "상영횟수",
    "screen_count": "스크린수",
    "country": "국적",
    "production_company": "제작사",
    "distributor": "배급사",
    "rating": "등급",
    "genre": "장르",
    "director": "감독",
    "actor": "배우",
    "synopsis": "시놉시스",
    "poster_url": "포스터URL"
}

df = df_raw.rename(columns=rename_map).copy()

print("\n컬럼명 통일 후 df shape:", df.shape)
print("\n컬럼명 통일 후 컬럼:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 5) 필수 컬럼 확인
# ------------------------------------------------------------

required_cols = [
    "영화명",
    "개봉일",
    "누적관객수",
    "국적",
    "제작사",
    "배급사",
    "등급",
    "장르",
    "감독",
    "배우",
    "시놉시스",
    "포스터URL"
]

missing_required_cols = [col for col in required_cols if col not in df.columns]

if len(missing_required_cols) > 0:
    raise ValueError(f"필수 컬럼이 없습니다: {missing_required_cols}")

print("\n필수 컬럼 확인 완료")


# ------------------------------------------------------------
# 6) 시놉시스 확률 컬럼 확인
# ------------------------------------------------------------

synopsis_prob_cols = [
    "시놉시스_prob_0",
    "시놉시스_prob_1",
    "시놉시스_prob_2",
    "시놉시스_prob_3"
]

missing_synopsis_prob_cols = [
    col for col in synopsis_prob_cols
    if col not in df.columns
]

if len(missing_synopsis_prob_cols) > 0:
    raise ValueError(f"시놉시스 확률 컬럼이 없습니다: {missing_synopsis_prob_cols}")

for col in synopsis_prob_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.25)

print("시놉시스 확률 컬럼 확인 완료:", synopsis_prob_cols)


# ------------------------------------------------------------
# 7) 기본 결측치 확인
# ------------------------------------------------------------

missing_summary = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_ratio"] = missing_summary["missing_count"] / len(df)

print("\n결측치 상위 20개:")
display(missing_summary.head(20))


# ------------------------------------------------------------
# 8) 전처리 로그 초기화
# ------------------------------------------------------------

preprocess_log = {
    "raw_csv_path": str(raw_csv_path),
    "raw_shape": tuple(df_raw.shape),
    "after_column_rename_shape": tuple(df.shape),
    "columns": df.columns.tolist(),
    "missing_summary_top20": missing_summary.head(20).to_dict(orient="records")
}

print("\n최종 df shape:", df.shape)
print("최종 컬럼:")
print(df.columns.tolist())

print("\nSTEP 1 수정판 완료")

# #2. 날짜 / 관객수 / 결측치 처리
* ? 누적 관객수 0 이하는 자르는데, 0 이하가 있나? 그냥 몇 명 이상 가져가는 거 아니었나?

In [ ]:
# ============================================================
# STEP 2. 기본 정제: 날짜 / 관객수 / 결측치 처리
# ============================================================

print("STEP 2 시작")

# ------------------------------------------------------------
# 1) STEP 2 시작 전 상태 저장
# ------------------------------------------------------------

before_step2_shape = df.shape

print("STEP 2 전 df shape:", before_step2_shape)


# ------------------------------------------------------------
# 2) 개봉일 datetime 변환
# ------------------------------------------------------------

df["개봉일"] = pd.to_datetime(df["개봉일"], errors="coerce")

invalid_date_count = df["개봉일"].isna().sum()

print("\n개봉일 datetime 변환 완료")
print("개봉일 변환 실패 행 수:", invalid_date_count)

# 개봉일이 없는 행 제거
before_drop_date = len(df)
df = df[df["개봉일"].notna()].copy()
after_drop_date = len(df)

print("개봉일 결측 제거 전:", before_drop_date)
print("개봉일 결측 제거 후:", after_drop_date)
print("제거된 행 수:", before_drop_date - after_drop_date)


# ------------------------------------------------------------
# 3) 분석 기간 필터링
# ------------------------------------------------------------

before_date_filter = len(df)

df = df[
    (df["개봉일"] >= START_DATE) &
    (df["개봉일"] <= END_DATE)
].copy()

after_date_filter = len(df)

print("\n분석 기간 필터링:", START_DATE.date(), "~", END_DATE.date())
print("필터링 전:", before_date_filter)
print("필터링 후:", after_date_filter)
print("제거된 행 수:", before_date_filter - after_date_filter)

print("\n필터링 후 개봉일 범위:")
print("최소 개봉일:", df["개봉일"].min())
print("최대 개봉일:", df["개봉일"].max())

print("\n연도별 영화 수:")
display(df["개봉일"].dt.year.value_counts().sort_index().reset_index().rename(
    columns={"개봉일": "개봉연도", "count": "영화수"}
))


# ------------------------------------------------------------
# 4) 누적관객수 숫자형 변환 및 0 이하 제거
# ------------------------------------------------------------

df["누적관객수"] = pd.to_numeric(df["누적관객수"], errors="coerce")

invalid_audience_count = df["누적관객수"].isna().sum()

print("\n누적관객수 숫자형 변환 완료")
print("누적관객수 변환 실패 행 수:", invalid_audience_count)

before_drop_audience = len(df)

df = df[
    df["누적관객수"].notna() &
    (df["누적관객수"] > 0)
].copy()

after_drop_audience = len(df)

print("누적관객수 결측/0 이하 제거 전:", before_drop_audience)
print("누적관객수 결측/0 이하 제거 후:", after_drop_audience)
print("제거된 행 수:", before_drop_audience - after_drop_audience)

df["누적관객수"] = df["누적관객수"].astype(int)


# ------------------------------------------------------------
# 5) 문자열 컬럼 결측치 처리
# ------------------------------------------------------------

text_cols = [
    "영화명",
    "국적",
    "제작사",
    "배급사",
    "등급",
    "장르",
    "감독",
    "배우",
    "시놉시스",
    "포스터URL"
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col] == "", col] = "Unknown"

print("\n문자열 컬럼 결측치 Unknown 처리 완료")


# ------------------------------------------------------------
# 6) 시놉시스 확률값 보정
# ------------------------------------------------------------
# 확률값이 모두 존재하지 않거나 합이 이상한 경우를 대비한다.
# 여기서는 결측은 0.25로 채우고, 합이 0이면 균등분포로 보정한다.
# 합이 0이 아닌 경우에는 합이 1이 되도록 정규화한다.

synopsis_prob_cols = [
    "시놉시스_prob_0",
    "시놉시스_prob_1",
    "시놉시스_prob_2",
    "시놉시스_prob_3"
]

for col in synopsis_prob_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.25)

prob_sum = df[synopsis_prob_cols].sum(axis=1)

zero_prob_mask = prob_sum <= 0

df.loc[zero_prob_mask, synopsis_prob_cols] = 0.25

prob_sum = df[synopsis_prob_cols].sum(axis=1)

df[synopsis_prob_cols] = df[synopsis_prob_cols].div(prob_sum, axis=0)

print("\n시놉시스 확률값 보정 완료")
print("확률합 최소:", df[synopsis_prob_cols].sum(axis=1).min())
print("확률합 최대:", df[synopsis_prob_cols].sum(axis=1).max())



# ------------------------------------------------------------
# 8) 중복 확인
# ------------------------------------------------------------
# 같은 영화명 + 같은 개봉일이 중복이면 우선 확인만 한다.
# 여기서 바로 제거하지는 않는다. 동일 제목 재개봉/동명이 있을 수 있기 때문.

duplicate_mask = df.duplicated(subset=["영화명", "개봉일"], keep=False)
duplicate_count = duplicate_mask.sum()

print("\n영화명 + 개봉일 기준 중복 행 수:", duplicate_count)

if duplicate_count > 0:
    print("중복 예시:")
    display(
        df.loc[duplicate_mask, ["영화명", "개봉일", "누적관객수", "배급사", "장르"]]
        .sort_values(["영화명", "개봉일"])
        .head(20)
    )


# ------------------------------------------------------------
# 9) index 정리
# ------------------------------------------------------------
# 주의:
# 포스터 embedding 파일이 기존 df index와 연결되어 있다면 reset_index가 문제될 수 있다.
# 하지만 공통 전처리 파일에서는 최종 processed data에 original_index를 저장해두고,
# 이후 CNN+MLP에서 poster_embedding_index.csv와 original_index 기준으로 연결할 수 있게 한다.

df["original_index"] = df.index

df = df.reset_index(drop=True)

print("\nindex 정리 완료")
print("original_index 컬럼 생성 후 reset_index 수행")


# ------------------------------------------------------------
# 10) STEP 2 로그 저장
# ------------------------------------------------------------

preprocess_log["step2"] = {
    "before_step2_shape": tuple(before_step2_shape),
    "after_step2_shape": tuple(df.shape),
    "invalid_date_count": int(invalid_date_count),
    "removed_by_date_missing": int(before_drop_date - after_drop_date),
    "removed_by_date_filter": int(before_date_filter - after_date_filter),
    "invalid_audience_count": int(invalid_audience_count),
    "removed_by_audience": int(before_drop_audience - after_drop_audience),
    "duplicate_movie_release_count": int(duplicate_count),
    "date_min": str(df["개봉일"].min()),
    "date_max": str(df["개봉일"].max())
}

print("\nSTEP 2 후 df shape:", df.shape)

print("\n결측치 상위 20개:")
display(
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)

print("\nSTEP 2 완료")

# #3. Target 생성 : 흥행 등급 / log 관객수

In [ ]:
# ============================================================
# STEP 3. Target 생성: 흥행 등급 / log 관객수
# ============================================================

print("STEP 3 시작")

# ------------------------------------------------------------
# 1) 흥행 등급 생성 함수
# ------------------------------------------------------------

def make_audience_class(audience):
    """
    누적관객수를 4개 흥행 등급으로 변환한다.

    0: 1만 미만
    1: 1만~10만
    2: 10만~100만
    3: 100만 이상
    """

    if audience < 10_000:
        return 0
    elif audience < 100_000:
        return 1
    elif audience < 1_000_000:
        return 2
    else:
        return 3


# ------------------------------------------------------------
# 2) y_class 생성(누적관객수 클래스)
# ------------------------------------------------------------

df["y_class"] = df["누적관객수"].apply(make_audience_class).astype(int)

df["y_class_name"] = df["y_class"].map(CLASS_BINS)

print("y_class 생성 완료")

print("\n흥행 등급 분포:")
class_dist = (
    df["y_class"]
    .value_counts()
    .sort_index()
    .reset_index()
)

class_dist.columns = ["y_class", "count"]
class_dist["class_name"] = class_dist["y_class"].map(CLASS_BINS)
class_dist["ratio"] = class_dist["count"] / len(df)

display(class_dist)


# ------------------------------------------------------------
# 3) y_reg_log 생성 (관객수 편차 커서 log 취함)
# ------------------------------------------------------------
# 관객수는 편차가 크므로 log1p 변환한다.
# 예: 1,000명과 10,000,000명의 차이가 너무 커서 원본값 그대로 회귀하면 대형 흥행작에 끌릴 수 있음.

df["y_reg_log"] = np.log1p(df["누적관객수"]).astype(float)

print("\ny_reg_log 생성 완료")

print("\ny_reg_log 통계:")
display(df["y_reg_log"].describe().reset_index().rename(
    columns={"index": "stat", "y_reg_log": "value"}
))


# ------------------------------------------------------------
# 4) 관객수 구간별 기본 통계 확인
# ------------------------------------------------------------

audience_summary_by_class = (
    df.groupby(["y_class", "y_class_name"])["누적관객수"]
    .agg(["count", "min", "median", "mean", "max"])
    .reset_index()
)

print("\n흥행 등급별 누적관객수 통계:")
display(audience_summary_by_class)


# ------------------------------------------------------------
# 5) target 결측 확인
# ------------------------------------------------------------

target_missing = df[["y_class", "y_class_name", "y_reg_log"]].isna().sum()

print("\ntarget 결측 확인:")
print(target_missing)

if target_missing.sum() > 0:
    raise ValueError("target 컬럼에 결측치가 있습니다.")


# ------------------------------------------------------------
# 6) STEP 3 로그 저장
# ------------------------------------------------------------

preprocess_log["step3"] = {
    "after_step3_shape": tuple(df.shape),
    "class_distribution": class_dist.to_dict(orient="records"),
    "y_reg_log_describe": df["y_reg_log"].describe().to_dict(),
    "audience_summary_by_class": audience_summary_by_class.to_dict(orient="records")
}

print("\nSTEP 3 후 df shape:", df.shape)
print("추가된 target 컬럼: y_class, y_class_name, y_reg_log")

print("\nSTEP 3 완료")

# #4. 공통 파생 feature 생성 - 날짜 / 제목 / 인원&기관 수 / 시놉시스 / 포스터 보유 여부

In [ ]:
# ============================================================
# STEP 4. 공통 파생 feature 생성
# - 날짜 feature
# - 제목 feature
# - 인원/기관 수 feature
# - 시놉시스 feature
# - 포스터 보유 여부 feature
# ============================================================

print("STEP 4 시작")

# ------------------------------------------------------------
# 1) 날짜 기반 feature
# ------------------------------------------------------------

df["개봉연도"] = df["개봉일"].dt.year.astype(int)
df["개봉월"] = df["개봉일"].dt.month.astype(int)
df["개봉분기"] = df["개봉일"].dt.quarter.astype(int)
df["개봉요일"] = df["개봉일"].dt.dayofweek.astype(int)  # 월=0, 일=6

# 금/토/일 개봉 여부
df["주말개봉여부"] = df["개봉요일"].isin([4, 5, 6]).astype(int)

# 한국 극장가 기준 성수기 단순 정의
# 여름방학/겨울방학/설·추석 근처를 대략 반영
df["성수기여부"] = df["개봉월"].isin([1, 2, 7, 8, 9, 12]).astype(int)

# 코로나 이후 여부
df["코로나이후여부"] = (df["개봉연도"] >= 2020).astype(int)

# 월별 공휴일/연휴 효과를 아주 단순하게 반영한 feature
# 정확한 공휴일 계산이 아니라, 월 단위 계절성 feature에 가까움
holiday_month_score = {
    1: 2,
    2: 2,
    3: 1,
    4: 0,
    5: 2,
    6: 1,
    7: 1,
    8: 2,
    9: 2,
    10: 2,
    11: 0,
    12: 2
}

df["휴일수"] = df["개봉월"].map(holiday_month_score).fillna(0).astype(int)

print("날짜 기반 feature 생성 완료")


# ------------------------------------------------------------
# 2) 제목 기반 feature
# ------------------------------------------------------------

df["영화명"] = df["영화명"].fillna("Unknown").astype(str)

df["제목길이"] = df["영화명"].str.len().astype(int)
df["제목_숫자포함"] = df["영화명"].str.contains(r"\d", regex=True).astype(int)
df["제목_콜론포함"] = df["영화명"].str.contains(r":|：", regex=True).astype(int)
df["제목_특수문자포함"] = df["영화명"].str.contains(r"[^가-힣a-zA-Z0-9\s]", regex=True).astype(int)

print("제목 기반 feature 생성 완료")


# ------------------------------------------------------------
# 3) 쉼표 기준 개수 feature 함수
# ------------------------------------------------------------

def count_items(text):
    """
    배우, 감독, 제작사, 배급사처럼 쉼표로 여러 값이 들어간 컬럼의 개수를 센다.
    Unknown, 결측, 빈 문자열은 0으로 처리한다.
    """

    if pd.isna(text):
        return 0

    text = str(text).strip()

    if text == "" or text == "Unknown" or text.lower() == "nan":
        return 0

    items = [
        item.strip()
        for item in re.split(r"[,/]", text)
        if item.strip() != ""
    ]

    return len(items)


# ------------------------------------------------------------
# 4) 시놉시스 기반 feature
# ------------------------------------------------------------

synopsis_prob_cols = [
    "시놉시스_prob_0",
    "시놉시스_prob_1",
    "시놉시스_prob_2",
    "시놉시스_prob_3"
]

# 혹시 모를 결측/비정상값 재보정
for col in synopsis_prob_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.25)

prob_sum = df[synopsis_prob_cols].sum(axis=1)
zero_prob_mask = prob_sum <= 0

df.loc[zero_prob_mask, synopsis_prob_cols] = 0.25
prob_sum = df[synopsis_prob_cols].sum(axis=1)
df[synopsis_prob_cols] = df[synopsis_prob_cols].div(prob_sum, axis=0)

# 시놉시스 확률에서 가장 높은 class
df["시놉시스_pred_class"] = df[synopsis_prob_cols].values.argmax(axis=1).astype(int)

# 가장 높은 확률값
df["시놉시스_confidence"] = df[synopsis_prob_cols].max(axis=1).astype(float)

# 시놉시스 텍스트 보유 여부
def has_valid_text(text):
    if pd.isna(text):
        return 0

    text = str(text).strip()

    if text == "" or text == "Unknown" or text.lower() == "nan":
        return 0

    return 1


df["시놉시스_보유여부"] = df["시놉시스"].apply(has_valid_text).astype(int)

print("시놉시스 feature 생성 완료")


# ------------------------------------------------------------
# 5) 포스터 URL 보유 여부 feature
# ------------------------------------------------------------

df["포스터_보유여부"] = df["포스터URL"].apply(has_valid_text).astype(int)

print("포스터 보유 여부 feature 생성 완료")


# ------------------------------------------------------------
# 6) feature 생성 결과 확인
# ------------------------------------------------------------

created_feature_cols_step4 = [
    "개봉연도",
    "개봉월",
    "개봉분기",
    "개봉요일",
    "주말개봉여부",
    "성수기여부",
    "코로나이후여부",
    "휴일수",
    "제목길이",
    "제목_숫자포함",
    "제목_콜론포함",
    "제목_특수문자포함",
    "시놉시스_pred_class",
    "시놉시스_confidence",
    "시놉시스_보유여부",
    "포스터_보유여부"
]

print("\nSTEP 4 생성 feature 목록:")
print(created_feature_cols_step4)

print("\n생성 feature 결측치 확인:")
display(
    df[created_feature_cols_step4]
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "feature", 0: "missing_count"})
)

print("\n생성 feature 기본 통계:")
display(df[created_feature_cols_step4].describe().T)


# ------------------------------------------------------------
# 7) 시놉시스/포스터 보유율 확인
# ------------------------------------------------------------

print("\n시놉시스 보유 여부 분포:")
display(df["시놉시스_보유여부"].value_counts().sort_index().reset_index().rename(
    columns={"시놉시스_보유여부": "value", "count": "count"}
))

print("\n포스터 URL 보유 여부 분포:")
display(df["포스터_보유여부"].value_counts().sort_index().reset_index().rename(
    columns={"포스터_보유여부": "value", "count": "count"}
))


# ------------------------------------------------------------
# 8) STEP 4 로그 저장
# ------------------------------------------------------------

preprocess_log["step4"] = {
    "after_step4_shape": tuple(df.shape),
    "created_feature_cols_step4": created_feature_cols_step4,
    "synopsis_available_distribution": df["시놉시스_보유여부"].value_counts().sort_index().to_dict(),
    "poster_available_distribution": df["포스터_보유여부"].value_counts().sort_index().to_dict()
}

print("\nSTEP 4 후 df shape:", df.shape)
print("\nSTEP 4 완료")

# #5. 장르 multi-hot / 주요 제작사+배급사+배우+감독 pool feature 생성

In [ ]:
# ============================================================
# STEP 5. 장르 multi-hot / 주요 pool feature 생성
# ============================================================

print("STEP 5 시작")

from sklearn.preprocessing import MultiLabelBinarizer


# ------------------------------------------------------------
# 1) 장르 문자열 정리 함수
# ------------------------------------------------------------

def split_genres(genre_text):
    """
    장르 문자열을 리스트로 변환한다.
    예:
    '드라마,멜로/로맨스' → ['드라마', '멜로']
    """

    if pd.isna(genre_text):
        return ["Unknown"]

    text = str(genre_text).strip()

    if text == "" or text == "Unknown" or text.lower() == "nan":
        return ["Unknown"]

    # 구분자 통일
    raw_items = re.split(r"[,/]", text)

    genre_list = []

    for item in raw_items:
        g = item.strip()

        if g == "":
            continue

        # 장르명 통일
        if g == "코메디":
            g = "코미디"

        if g in ["로맨스", "멜로/로맨스"]:
            g = "멜로"

        genre_list.append(g)

    if len(genre_list) == 0:
        return ["Unknown"]

    return sorted(list(set(genre_list)))


# ------------------------------------------------------------
# 2) 장르 리스트 생성
# ------------------------------------------------------------

df["장르_list"] = df["장르"].apply(split_genres)

print("장르_list 생성 완료")

print("\n장르_list 예시:")
display(df[["영화명", "장르", "장르_list"]].head(10))


# ------------------------------------------------------------
# 3) MultiLabelBinarizer로 장르 multi-hot 생성
# ------------------------------------------------------------

mlb = MultiLabelBinarizer()

genre_matrix = mlb.fit_transform(df["장르_list"])

genre_cols = [f"장르_{genre}" for genre in mlb.classes_]

genre_df = pd.DataFrame(
    genre_matrix,
    columns=genre_cols,
    index=df.index
)

df = pd.concat([df, genre_df], axis=1)

print("\n장르 multi-hot 생성 완료")
print("장르 feature 개수:", len(genre_cols))
print("장르 feature 목록:")
print(genre_cols)

print("\n장르별 영화 수:")
display(
    df[genre_cols]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "genre_feature", 0: "count"})
)


# ------------------------------------------------------------
# 4) 인기 배우/감독/제작사/배급사 feature 파일 로드
# ------------------------------------------------------------

POPULAR_ENTITY_PATH = DATA_DIR / "popular_entity_features_utf8_sig.csv"

if not POPULAR_ENTITY_PATH.exists():
    raise FileNotFoundError(f"popular entity feature 파일을 찾을 수 없습니다: {POPULAR_ENTITY_PATH}")

popular_entity_df = pd.read_csv(POPULAR_ENTITY_PATH)

print("\npopular_entity_df 로드 완료")
print("popular_entity_df shape:", popular_entity_df.shape)
print("popular_entity_df columns:")
print(popular_entity_df.columns.tolist())

display(popular_entity_df.head())


# ------------------------------------------------------------
# 5) merge key 정리
# ------------------------------------------------------------

popular_entity_df["release_date"] = pd.to_datetime(
    popular_entity_df["release_date"],
    errors="coerce"
)

# df 쪽 merge key
df["merge_movie_name"] = df["영화명"].astype(str).str.strip()
df["merge_release_date"] = pd.to_datetime(df["개봉일"], errors="coerce")

# popular entity 쪽 merge key
popular_entity_df["merge_movie_name"] = popular_entity_df["movie_name_clean"].astype(str).str.strip()
popular_entity_df["merge_release_date"] = popular_entity_df["release_date"]


# ------------------------------------------------------------
# 6) 사용할 popular entity feature 목록
# ------------------------------------------------------------

popular_entity_feature_cols = [
    "has_popular_director",
    "popular_director_count",
    "has_popular_actor",
    "popular_actor_count",
    "top_popular_actor_mean_audience",
    "has_popular_production_company",
    "popular_production_company_count",
    "has_popular_distributor",
    "popular_distributor_count"
]

missing_popular_cols = [
    col for col in popular_entity_feature_cols
    if col not in popular_entity_df.columns
]

if len(missing_popular_cols) > 0:
    raise ValueError(f"popular entity 파일에 필요한 컬럼이 없습니다: {missing_popular_cols}")


# ------------------------------------------------------------
# 7) df에 popular entity feature 병합
# ------------------------------------------------------------

before_merge_shape = df.shape

df = df.merge(
    popular_entity_df[
        ["merge_movie_name", "merge_release_date"] + popular_entity_feature_cols
    ],
    on=["merge_movie_name", "merge_release_date"],
    how="left"
)

after_merge_shape = df.shape

print("\npopular entity feature merge 완료")
print("merge 전 df shape:", before_merge_shape)
print("merge 후 df shape:", after_merge_shape)


# ------------------------------------------------------------
# 8) merge 실패한 행은 0으로 처리
# ------------------------------------------------------------

for col in popular_entity_feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# 여부 feature는 int로 정리
binary_popular_cols = [
    "has_popular_director",
    "has_popular_actor",
    "has_popular_production_company",
    "has_popular_distributor"
]

for col in binary_popular_cols:
    df[col] = df[col].astype(int)

# count feature도 int로 정리
count_popular_cols = [
    "popular_director_count",
    "popular_actor_count",
    "popular_production_company_count",
    "popular_distributor_count"
]

for col in count_popular_cols:
    df[col] = df[col].astype(int)

# 평균 관객수 feature는 float 유지
df["top_popular_actor_mean_audience"] = df["top_popular_actor_mean_audience"].astype(float)

print("\npopular entity feature 결측치 0 처리 완료")


# ------------------------------------------------------------
# 9) merge 결과 확인
# ------------------------------------------------------------

print("\npopular entity feature 분포/통계:")

display(df[popular_entity_feature_cols].describe().T)

print("\nmerge된 popular entity feature 결측치:")
display(
    df[popular_entity_feature_cols]
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "feature", 0: "missing_count"})
)


# ------------------------------------------------------------
# 10) 임시 merge key 제거
# ------------------------------------------------------------

df = df.drop(columns=["merge_movie_name", "merge_release_date"])

print("\n임시 merge key 제거 완료")


# ------------------------------------------------------------
# 11) STEP 5 생성 feature 정리
# ------------------------------------------------------------

created_feature_cols_step5 = genre_cols + popular_entity_feature_cols

preprocess_log["step5"] = {
    "after_step5_shape": tuple(df.shape),
    "genre_cols": genre_cols,
    "n_genre_cols": len(genre_cols),
    "popular_entity_path": str(POPULAR_ENTITY_PATH),
    "popular_entity_feature_cols": popular_entity_feature_cols
}

print("\nSTEP 5 후 df shape:", df.shape)
print("장르 feature 수:", len(genre_cols))
print("popular entity feature 수:", len(popular_entity_feature_cols))

print("\nSTEP 5 완료")

# #6. 최종 공통 feature

In [ ]:
# ============================================================
# STEP 6. 최종 공통 feature 목록 구성 / 누수 feature 제거
# ============================================================

print("STEP 6 시작")


# ------------------------------------------------------------
# 1) 숫자형 feature 목록 정의
# ------------------------------------------------------------

numeric_features = [
    # 날짜 feature
    "개봉연도",
    "개봉월",
    "개봉분기",
    "개봉요일",
    "주말개봉여부",
    "성수기여부",
    "코로나이후여부",
    "휴일수",

    # 제목 feature
    "제목길이",
    "제목_숫자포함",
    "제목_콜론포함",
    "제목_특수문자포함",

    # 시놉시스 확률 feature
    "시놉시스_prob_0",
    "시놉시스_prob_1",
    "시놉시스_prob_2",
    "시놉시스_prob_3",
    "시놉시스_pred_class",
    "시놉시스_confidence",
    "시놉시스_보유여부",

    # 포스터 URL 보유 여부
    "포스터_보유여부",

    # 인기 entity feature
    "has_popular_director",
    "popular_director_count",
    "has_popular_actor",
    "popular_actor_count",
    "top_popular_actor_mean_audience",
    "has_popular_production_company",
    "popular_production_company_count",
    "has_popular_distributor",
    "popular_distributor_count",
]


# ------------------------------------------------------------
# 2) 범주형 feature 목록 정의
# ------------------------------------------------------------
# OneHotEncoder 또는 LightGBM categorical feature로 사용할 수 있는 원본 범주형 컬럼

categorical_features = [
    "국적",
    "등급",
    "제작사",
    "배급사",
    "감독"
]


# ------------------------------------------------------------
# 3) 장르 multi-hot feature 목록 확인
# ------------------------------------------------------------

# STEP 5에서 genre_cols가 생성되어 있어야 한다.
if "genre_cols" not in globals():
    raise ValueError("genre_cols가 없습니다. STEP 5를 먼저 실행하세요.")

print("장르 feature 수:", len(genre_cols))


# ------------------------------------------------------------
# 4) 누수 feature 목록 정의
# ------------------------------------------------------------
# 개봉 후에야 알 수 있는 정보는 개봉 예정작 예측에 사용할 수 없으므로 feature에서 제외한다.

leakage_features = [
    "누적관객수",
    "상영횟수",
    "스크린수",
    "show_count",
    "screen_count",
    "y_class",
    "y_class_name",
    "y_reg_log"
]

print("\n누수 feature 목록:")
print(leakage_features)


# ------------------------------------------------------------
# 5) 최종 feature_cols 구성
# ------------------------------------------------------------

feature_cols = numeric_features + categorical_features + genre_cols

# 중복 제거: 순서 유지
feature_cols = list(dict.fromkeys(feature_cols))

print("\n최종 feature_cols 개수:", len(feature_cols))


# ------------------------------------------------------------
# 6) feature 존재 여부 확인
# ------------------------------------------------------------

missing_numeric_features = [col for col in numeric_features if col not in df.columns]
missing_categorical_features = [col for col in categorical_features if col not in df.columns]
missing_genre_features = [col for col in genre_cols if col not in df.columns]
missing_feature_cols = [col for col in feature_cols if col not in df.columns]

print("\nmissing_numeric_features:", missing_numeric_features)
print("missing_categorical_features:", missing_categorical_features)
print("missing_genre_features:", missing_genre_features)
print("missing_feature_cols:", missing_feature_cols)

if len(missing_feature_cols) > 0:
    raise ValueError(f"최종 feature_cols 중 df에 없는 컬럼이 있습니다: {missing_feature_cols}")

print("\n최종 feature 존재 여부 확인 완료")


# ------------------------------------------------------------
# 7) 누수 feature가 feature_cols에 들어갔는지 검사
# ------------------------------------------------------------

leakage_in_features = [col for col in leakage_features if col in feature_cols]

print("\nfeature_cols에 포함된 누수 feature:")
print(leakage_in_features)

if not USE_LEAKAGE_FEATURES and len(leakage_in_features) > 0:
    raise ValueError(f"누수 feature가 feature_cols에 포함되어 있습니다: {leakage_in_features}")

print("누수 feature 검사 완료")


# ------------------------------------------------------------
# 8) 최종 feature 결측치 확인
# ------------------------------------------------------------

feature_missing_summary = (
    df[feature_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

feature_missing_summary.columns = ["feature", "missing_count"]
feature_missing_summary["missing_ratio"] = feature_missing_summary["missing_count"] / len(df)

print("\n최종 feature 결측치 상위 30개:")
display(feature_missing_summary.head(30))


# ------------------------------------------------------------
# 9) numeric feature 숫자형 보정
# ------------------------------------------------------------

for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print("\nnumeric feature 숫자형 보정 완료")


# ------------------------------------------------------------
# 10) categorical feature 문자열 보정
# ------------------------------------------------------------

for col in categorical_features:
    df[col] = df[col].fillna("Unknown").astype(str).str.strip()
    df.loc[df[col] == "", col] = "Unknown"

print("categorical feature 문자열 보정 완료")


# ------------------------------------------------------------
# 11) genre feature 정수형 보정
# ------------------------------------------------------------

for col in genre_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

print("genre feature 정수형 보정 완료")


# ------------------------------------------------------------
# 12) feature dtype 확인
# ------------------------------------------------------------

feature_dtype_df = pd.DataFrame({
    "feature": feature_cols,
    "dtype": [str(df[col].dtype) for col in feature_cols],
    "missing_count": [int(df[col].isna().sum()) for col in feature_cols],
    "nunique": [int(df[col].nunique()) for col in feature_cols]
})

print("\n최종 feature dtype 요약:")
display(feature_dtype_df)


# ------------------------------------------------------------
# 13) target 컬럼 목록
# ------------------------------------------------------------

target_cols = [
    "누적관객수",
    "y_class",
    "y_class_name",
    "y_reg_log"
]

id_cols = [
    "original_index",
    "영화명",
    "개봉일"
]

auxiliary_cols = [
    "시놉시스",
    "포스터URL",
    "장르",
    "장르_list",
    "배우",
    "제작사",
    "배급사",
    "감독"
]

# 저장 시 포함할 컬럼
# id + target + feature + 보조 설명 컬럼 일부
save_cols = []

for col in id_cols + target_cols + feature_cols + auxiliary_cols:
    if col in df.columns and col not in save_cols:
        save_cols.append(col)

print("\n저장 대상 컬럼 수:", len(save_cols))


# ------------------------------------------------------------
# 14) STEP 6 로그 저장
# ------------------------------------------------------------

preprocess_log["step6"] = {
    "after_step6_shape": tuple(df.shape),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "genre_cols": genre_cols,
    "feature_cols": feature_cols,
    "n_feature_cols": len(feature_cols),
    "leakage_features": leakage_features,
    "leakage_in_features": leakage_in_features,
    "target_cols": target_cols,
    "id_cols": id_cols,
    "auxiliary_cols": auxiliary_cols,
    "save_cols": save_cols
}

print("\nSTEP 6 후 df shape:", df.shape)
print("numeric_features:", len(numeric_features))
print("categorical_features:", len(categorical_features))
print("genre_cols:", len(genre_cols))
print("feature_cols:", len(feature_cols))

print("\nSTEP 6 완료")

# #7. 최종 저장 단계

In [ ]:
# ============================================================
# STEP 7. 공통 전처리 결과 저장
# ============================================================

print("STEP 7 시작")

# ------------------------------------------------------------
# 1) 저장 전 최종 데이터 확인
# ------------------------------------------------------------

print("최종 df shape:", df.shape)
print("저장 대상 컬럼 수:", len(save_cols))
print("최종 feature 수:", len(feature_cols))

print("\n최종 저장 대상 컬럼:")
print(save_cols)

# 저장할 데이터프레임 생성
processed_df = df[save_cols].copy()

print("\nprocessed_df shape:", processed_df.shape)

display(processed_df.head())


# ------------------------------------------------------------
# 2) 최종 결측치 점검
# ------------------------------------------------------------

final_missing_summary = (
    processed_df
    .isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

final_missing_summary.columns = ["column", "missing_count"]
final_missing_summary["missing_ratio"] = final_missing_summary["missing_count"] / len(processed_df)

print("\n최종 저장 데이터 결측치 상위 30개:")
display(final_missing_summary.head(30))

# feature_cols에는 결측이 없어야 함
feature_missing_count = processed_df[feature_cols].isna().sum().sum()

print("\nfeature_cols 전체 결측치 개수:", feature_missing_count)

if feature_missing_count > 0:
    raise ValueError("feature_cols 안에 결측치가 남아 있습니다.")

print("feature_cols 결측치 없음 확인 완료")


# ------------------------------------------------------------
# 3) target 분포 최종 확인
# ------------------------------------------------------------

print("\ny_class 최종 분포:")
final_class_dist = (
    processed_df["y_class"]
    .value_counts()
    .sort_index()
    .reset_index()
)

final_class_dist.columns = ["y_class", "count"]
final_class_dist["class_name"] = final_class_dist["y_class"].map(CLASS_BINS)
final_class_dist["ratio"] = final_class_dist["count"] / len(processed_df)

display(final_class_dist)

print("\ny_reg_log 최종 통계:")
display(processed_df["y_reg_log"].describe().reset_index().rename(
    columns={"index": "stat", "y_reg_log": "value"}
))


# ------------------------------------------------------------
# 4) feature_info 저장용 dict 생성
# ------------------------------------------------------------

feature_info = {
    # 기본 설정
    "SEED": SEED,
    "START_DATE": str(START_DATE.date()),
    "END_DATE": str(END_DATE.date()),
    "CLASS_BINS": CLASS_BINS,
    "USE_LEAKAGE_FEATURES": USE_LEAKAGE_FEATURES,

    # 컬럼 목록
    "id_cols": id_cols,
    "target_cols": target_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "genre_cols": genre_cols,
    "feature_cols": feature_cols,
    "auxiliary_cols": auxiliary_cols,
    "save_cols": save_cols,

    # feature 개수
    "n_numeric_features": len(numeric_features),
    "n_categorical_features": len(categorical_features),
    "n_genre_cols": len(genre_cols),
    "n_feature_cols": len(feature_cols),

    # target 정보
    "class_name_map": CLASS_BINS,
    "n_classes": len(CLASS_BINS),

    # CNN+MLP 연결용 정보
    # poster_embedding_index.csv의 index는 여기 original_index와 매칭하면 됨
    "poster_join_key": "original_index",

    # 시놉시스 관련 정보
    "synopsis_prob_cols": [
        "시놉시스_prob_0",
        "시놉시스_prob_1",
        "시놉시스_prob_2",
        "시놉시스_prob_3"
    ],

    # 누수 feature
    "leakage_features": leakage_features
}


# ------------------------------------------------------------
# 5) preprocess_log 최종 업데이트
# ------------------------------------------------------------

preprocess_log["step7"] = {
    "final_df_shape": tuple(df.shape),
    "processed_df_shape": tuple(processed_df.shape),
    "processed_data_path": str(PROCESSED_DATA_PATH),
    "feature_info_path": str(FEATURE_INFO_PATH),
    "preprocess_log_path": str(PREPROCESS_LOG_PATH),
    "final_class_distribution": final_class_dist.to_dict(orient="records"),
    "feature_missing_count": int(feature_missing_count),
    "n_feature_cols": len(feature_cols),
    "n_save_cols": len(save_cols)
}


# ------------------------------------------------------------
# 6) 파일 저장
# ------------------------------------------------------------

processed_df.to_csv(
    PROCESSED_DATA_PATH,
    index=False,
    encoding="utf-8-sig"
)

joblib.dump(
    feature_info,
    FEATURE_INFO_PATH
)

with open(PREPROCESS_LOG_PATH, "w", encoding="utf-8") as f:
    json.dump(
        preprocess_log,
        f,
        ensure_ascii=False,
        indent=2,
        default=str
    )

print("\n저장 완료:")
print("processed data:", PROCESSED_DATA_PATH)
print("feature info:", FEATURE_INFO_PATH)
print("preprocess log:", PREPROCESS_LOG_PATH)


# ------------------------------------------------------------
# 7) 저장 파일 다시 로드해서 검증
# ------------------------------------------------------------

loaded_processed_df = pd.read_csv(PROCESSED_DATA_PATH)
loaded_feature_info = joblib.load(FEATURE_INFO_PATH)

print("\n저장 파일 재로드 검증:")
print("loaded_processed_df shape:", loaded_processed_df.shape)
print("loaded feature_cols 수:", len(loaded_feature_info["feature_cols"]))
print("loaded numeric_features 수:", len(loaded_feature_info["numeric_features"]))
print("loaded categorical_features 수:", len(loaded_feature_info["categorical_features"]))
print("loaded genre_cols 수:", len(loaded_feature_info["genre_cols"]))

if loaded_processed_df.shape != processed_df.shape:
    raise ValueError("저장 후 다시 불러온 processed_df shape가 다릅니다.")

if loaded_feature_info["feature_cols"] != feature_cols:
    raise ValueError("저장 후 다시 불러온 feature_cols가 다릅니다.")

print("\n저장 파일 검증 완료")


# ------------------------------------------------------------
# 8) 이후 모델 파일에서 불러오는 예시 출력
# ------------------------------------------------------------

## 해당 내용은 실행환경에 따라 달라질 수 있음.
print("\n이후 모델 파일에서 사용하는 예시 코드:")
print("""
import pandas as pd
import joblib
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/흥보위")
COMMON_DIR = BASE_DIR / "공통 데이터 전처리"

PROCESSED_DATA_PATH = COMMON_DIR / "movie_common_processed.csv"
FEATURE_INFO_PATH = COMMON_DIR / "movie_common_feature_info.pkl"

df = pd.read_csv(PROCESSED_DATA_PATH)
feature_info = joblib.load(FEATURE_INFO_PATH)

numeric_features = feature_info["numeric_features"]
categorical_features = feature_info["categorical_features"]
genre_cols = feature_info["genre_cols"]
feature_cols = feature_info["feature_cols"]

X = df[feature_cols].copy()
y_class = df["y_class"].astype(int)
y_reg_log = df["y_reg_log"].astype(float)
""")

print("\nSTEP 7 완료")